<a href="https://colab.research.google.com/github/omsawarkar/omsawarkar/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omsawarkar/omsawarkar/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Signal checks and verdicts

Signal 1 — CONFIRMED: CTR differs across position buckets, supporting the position-aware reasoning behind FlyRank’s CTR-fix logic. Pooled CTR falls from 0.778% at positions 1–3 to 0.702%, 0.547%, and 0.205% in progressively worse buckets. These are observed associations, not causal effects.

Signal 2 — OPPOSITE: Higher impression volume does not correspond to higher median CTR within position buckets. Median CTR decreases across the three volume groups in all four position buckets. For example, at positions >3–10 it falls from 1.141% to 0.996% to 0.612%. Volume will represent exposure, not evidence of better performance.

### Proposed baseline rule

Using only 1–20 March 2026 observations, I will compare each eligible item’s CTR with the median CTR of its position-and-volume peer group. The review score is impressions multiplied by the positive gap between peer-median CTR and the item’s CTR.

Scoring requires at least 10 observed days, both measurement sources available, at least 100 impressions, a valid position, and at least 20 items in the peer group. These are provisional thresholds. Items outside these conditions will receive score zero and an insufficient-data reason.

A positive score produces reason code BELOW_PEER_CTR and action REVIEW_SEARCH_APPEARANCE_AND_INTENT. An eligible item without a positive gap receives NO_CTR_GAP and action MONITOR. Items without enough evidence receive INSUFFICIENT_DATA and action COLLECT_MORE_DATA.

The score is a descriptive peer-gap index, not expected recoverable clicks or proof that an update will help. Review may identify title, snippet, query-intent, or content issues, but no automatic edits are justified.

The conceptual decision point is after the early-window data becomes available. Later March outcomes and June data are excluded from the score. This baseline examines CTR review opportunities within my broader content-review lane; it does not use content age because this daily table does not provide update history.

In [6]:
%pip -q install huggingface_hub duckdb

from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

# Read the existing token privately.
token = userdata.get("HF_TOKEN").strip()

# Use the same March partition as Week 3.
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=(
        "fact_content_daily_performance/"
        "month=2026-03/data_0.parquet"
    ),
    repo_type="dataset",
    token=token
)

con = duckdb.connect()
con.read_parquet(
    march_path,
    hive_partitioning=True
).create_view("march_raw")

print("March dataset ready.")

display(con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM march_raw
""").df())


March dataset ready.


,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [7]:
# Use only the early window, matching the Week 3 inputs.
pages = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS observed_days,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(ga4_sessions) AS sessions,
        SUM(ga4_engaged_sessions) * 1.0
            / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,

        SUM(
            CASE WHEN gsc_avg_position > 0
                 THEN gsc_avg_position * gsc_impressions END
        ) / NULLIF(
            SUM(
                CASE WHEN gsc_avg_position > 0
                     THEN gsc_impressions END
            ), 0
        ) AS avg_position

    FROM march_raw
    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-20'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING COUNT(*) >= 10
       AND SUM(gsc_impressions) > 0
""").df()

import pandas as pd

pages["position_bucket"] = pd.cut(
    pages["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1–3", ">3–10", ">10–20", ">20"]
)

# Use a provisional volume floor to reduce very small samples.
signal_sample = pages.loc[
    (pages["impressions"] >= 100)
    & pages["position_bucket"].notna()
].copy()

position_table = (
    signal_sample.groupby("position_bucket", observed=True)
    .agg(
        item_count=("impressions", "size"),
        total_impressions=("impressions", "sum"),
        total_clicks=("clicks", "sum"),
        median_item_ctr=("ctr", "median")
    )
)

position_table["pooled_ctr_pct"] = (
    100 * position_table["total_clicks"]
    / position_table["total_impressions"]
)
position_table["median_item_ctr_pct"] = (
    100 * position_table.pop("median_item_ctr")
)

print("Signal 1: CTR compared across position buckets")
print("Input window: 1–20 March 2026")
display(position_table.round(3))

Signal 1: CTR compared across position buckets
Input window: 1–20 March 2026


,item_count,total_impressions,total_clicks,pooled_ctr_pct,median_item_ctr_pct
position_bucket,,,,,
1–3,292,2246189.0,17481.0,0.778,0.818
>3–10,2036,9210735.0,64622.0,0.702,0.746
>10–20,1683,6166116.0,33715.0,0.547,0.557
>20,2568,15775592.0,32324.0,0.205,0.189


In [8]:
# Compare volume groups within the same position buckets.
volume_sample = signal_sample.copy()

volume_sample["volume_bucket"] = pd.cut(
    volume_sample["impressions"],
    bins=[99, 499, 1999, float("inf")],
    labels=["100–499", "500–1,999", "2,000+"]
)

volume_table = (
    volume_sample.groupby(
        ["position_bucket", "volume_bucket"],
        observed=True
    )
    .agg(
        item_count=("impressions", "size"),
        total_impressions=("impressions", "sum"),
        total_clicks=("clicks", "sum"),
        median_item_ctr=("ctr", "median")
    )
)

volume_table["pooled_ctr_pct"] = (
    100 * volume_table["total_clicks"]
    / volume_table["total_impressions"]
)

volume_table["median_item_ctr_pct"] = (
    100 * volume_table.pop("median_item_ctr")
)

print("Signal 2: volume versus CTR within position buckets")
display(volume_table.round(3))

Signal 2: volume versus CTR within position buckets


item_count  total_impressions  total_clicks  \
position_bucket volume_bucket                                                
1–3             100–499                14             4591.0         101.0   
                500–1,999              82           103056.0        1265.0   
                2,000+                196          2138542.0       16115.0   
>3–10           100–499               138            44565.0         708.0   
                500–1,999             761           905790.0       10125.0   
                2,000+               1137          8260380.0       53789.0   
>10–20          100–499               134            44505.0         448.0   
                500–1,999             709           851315.0        6226.0   
                2,000+                840          5270296.0       27041.0   
>20             100–499                89            27981.0         160.0   
                500–1,999             697           895254.0        3545.0   
                2,000+               1782         14852357.0       28619.0   

                               pooled_ctr_pct  median_item_ctr_pct  
position_bucket volume_bucket                                       
1–3             100–499                 2.200                1.345  
                500–1,999               1.227                1.147  
                2,000+                  0.754                0.693  
>3–10           100–499                 1.589                1.141  
                500–1,999               1.118                0.996  
                2,000+                  0.651                0.612  
>10–20          100–499                 1.007                0.782  
                500–1,999               0.731                0.615  
                2,000+                  0.513                0.496  
>20             100–499                 0.572                0.283  
                500–1,999               0.396                0.278  
                2,000+                  0.193                0.159

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
from pathlib import Path
import numpy as np

queue = pages.copy()

# Define the same volume groups used in the signal check.
queue["volume_bucket"] = pd.cut(
    queue["impressions"],
    bins=[99, 499, 1999, float("inf")],
    labels=["100–499", "500–1,999", "2,000+"]
)

# Calculate peer benchmarks from eligible early-window items.
peer_stats = (
    queue.loc[
        (queue["impressions"] >= 100)
        & queue["position_bucket"].notna()
    ]
    .groupby(
        ["position_bucket", "volume_bucket"],
        observed=True
    )
    .agg(
        peer_count=("ctr", "size"),
        peer_median_ctr=("ctr", "median")
    )
    .reset_index()
)

queue = queue.merge(
    peer_stats,
    on=["position_bucket", "volume_bucket"],
    how="left",
    validate="many_to_one"
)

eligible = (
    (queue["observed_days"] >= 10)
    & (queue["impressions"] >= 100)
    & queue["position_bucket"].notna()
    & (queue["peer_count"] >= 20)
    & queue["ctr"].notna()
    & queue["peer_median_ctr"].notna()
)

queue["ctr_gap"] = (
    queue["peer_median_ctr"] - queue["ctr"]
).clip(lower=0)

queue["score"] = np.where(
    eligible,
    queue["impressions"] * queue["ctr_gap"],
    0.0
)

queue["reason_code"] = np.select(
    [~eligible, queue["score"] > 0],
    ["INSUFFICIENT_DATA", "BELOW_PEER_CTR"],
    default="NO_CTR_GAP"
)

queue["action"] = queue["reason_code"].map({
    "INSUFFICIENT_DATA": "COLLECT_MORE_DATA",
    "BELOW_PEER_CTR": "REVIEW_SEARCH_APPEARANCE_AND_INTENT",
    "NO_CTR_GAP": "MONITOR"
})

queue["confidence_note"] = np.where(
    eligible,
    "Descriptive peer comparison; action benefit unverified",
    "Insufficient evidence for scoring"
)

# Sort with deterministic tie-breaking.
queue = queue.sort_values(
    ["score", "impressions", "client_hash_id", "content_hash_id"],
    ascending=[False, False, True, True]
).reset_index(drop=True)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

# Check that the queue has one row per client–content pair.
assert not queue.duplicated(
    ["client_hash_id", "content_hash_id"]
).any()
assert queue["score"].notna().all()
assert (queue["score"] >= 0).all()

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "baseline_action_score.csv"
queue.to_csv(csv_path, index=False)

print("Queue items:", len(queue))
print("Positive-score items:", (queue["score"] > 0).sum())
print("CSV written:", csv_path)

# Show the top ten without displaying identifiers.
display(queue[[
    "rank",
    "score",
    "impressions",
    "clicks",
    "position_bucket",
    "volume_bucket",
    "ctr",
    "peer_median_ctr",
    "peer_count",
    "reason_code",
    "action"
]].head(10).round(5))


Queue items: 6607
Positive-score items: 3280
CSV written: work/outputs/baseline_action_score.csv


,rank,score,impressions,clicks,position_bucket,volume_bucket,ctr,peer_median_ctr,peer_count,reason_code,action
0,1,407.94216,176168.0,466.0,>10–20,"2,000+",0.00265,0.00496,840.0,BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT
1,2,344.42396,75879.0,32.0,>10–20,"2,000+",0.00042,0.00496,840.0,BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT
2,3,292.60369,85591.0,132.0,>10–20,"2,000+",0.00154,0.00496,840.0,BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT
3,4,275.44197,51074.0,37.0,>3–10,"2,000+",0.00072,0.00612,1137.0,BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT
4,5,249.47069,42088.0,8.0,>3–10,"2,000+",0.00019,0.00612,1137.0,BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT
5,6,216.74191,41315.0,36.0,>3–10,"2,000+",0.00087,0.00612,1137.0,BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT
6,7,197.10221,53307.0,129.0,>3–10,"2,000+",0.00242,0.00612,1137.0,BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT
7,8,188.12351,48243.0,107.0,>3–10,"2,000+",0.00222,0.00612,1137.0,BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT
8,9,181.57540,47663.0,110.0,>3–10,"2,000+",0.00231,0.00612,1137.0,BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT
9,10,181.38147,36679.0,43.0,>3–10,"2,000+",0.00117,0.00612,1137.0,BELOW_PEER_CTR,REVIEW_SEARCH_APPEARANCE_AND_INTENT


## 3. Top-20 review

This is a desk review of the numerical evidence, not an inspection of the actual pages. All ten items have reason code BELOW_PEER_CTR. The suggested action is to review search appearance and query intent before deciding whether any content change is appropriate. Confidence in an update producing benefits remains unverified.

| Rank | Suggested action and why it appears | What could make this recommendation wrong |
|---|---|---|
| 1 | Review search appearance and intent. It has 176,168 impressions, 468 clicks, and CTR below its >10–20 position peer benchmark, giving the largest score. | High exposure drives its rank; its query mix or search-result layout may naturally produce lower CTR than its peers. |
| 2 | Investigate the unusually low click rate: 32 clicks from 75,879 impressions in the >10–20 position bucket. | Tracking or aggregation problems could explain the low rate. Check measurement quality before recommending edits. |
| 3 | Review intent alignment and search appearance. It has 85,591 impressions and 132 clicks, below its >10–20 peer benchmark. | Its average position may hide a different distribution of query positions from the peer group. |
| 4 | Review search appearance. It has 51,074 impressions and 37 clicks despite an average position in the >3–10 bucket. | Branded versus non-branded queries, device mix, or search features could explain the gap without a content problem. |
| 5 | Investigate the very low click rate: 8 clicks from 42,088 impressions in the >3–10 bucket. | This is a weak pick until measurement quality and irrelevant-query exposure are checked; a low CTR alone does not justify an update. |
| 6 | Review query relevance and search appearance. It has 41,315 impressions and 36 clicks, below its >3–10 peer benchmark. | The broad peer bucket may compare it with pages that serve very different audiences or purposes. |
| 7 | Review search appearance and intent. It has 53,307 impressions and 129 clicks, creating a substantial peer gap. | Its lower CTR may reflect legitimate differences in search intent rather than an editable page issue. |
| 8 | Review whether the search result accurately represents the page. It has 48,243 impressions and 107 clicks in the >3–10 bucket. | Daily position or traffic variation may be hidden by the aggregated early-window values. |
| 9 | Review search appearance and intent. It has 47,863 impressions and 110 clicks, below the >3–10 benchmark. | The benchmark mixes clients, so client-specific search patterns may make this comparison unsuitable. |
| 10 | Review query relevance before considering changes. It has 36,679 impressions and 43 clicks in the >3–10 bucket. | Different observation-day coverage could affect its exposure total and ranking compared with other items. |

None of these reviews establishes that a recommendation is correct. Human-review Precision@10 cannot be measured without independent page-level review labels. The peer-gap score is not a prediction of recoverable clicks.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

### Weak picks and limitations

Ranks 2 and 5 deserve particular caution because they have very few clicks relative to impressions. Measurement issues or irrelevant-query exposure could explain their low CTR. They warrant investigation rather than automatic content edits.

The peer benchmarks mix clients and use broad position and volume buckets. Query intent, device, country, search-result layout, and differences in observation coverage remain uncontrolled. Each item also contributes to its own peer median. The score identifies an observed gap, not the cause of that gap or recoverable clicks.

### Leakage check

Scoring uses only observations dated 1–20 March 2026. No later-March outcomes, June observations, outcome labels, or product-decision flags enter the score. Client and content IDs are used only for grouping, identifying recommendations, and deterministic tie-breaking.

The peer benchmarks are calculated from the same early-window reference population. They are descriptive benchmarks, not an independent evaluation. Before comparing a trained model with this baseline, the benchmark procedure, eligible population, outcome definition, and evaluation split must be fixed consistently.

### Output and evaluation

The notebook regenerates work/outputs/baseline_action_score.csv. The CSV remains outside GitHub. A JSON run receipt records the configuration and summary counts.

The queue contains 6,607 items, including 3,280 with positive scores. No independent human-review labels are available, so Precision@10 and its corresponding review-label base rate remain unmeasured. No claim of validated action quality is made.

In [11]:
import json
from datetime import datetime, timezone

# Check the actual input window.
input_check = con.sql("""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(*) AS available_daily_rows
    FROM march_raw
    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-20'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
""").df()

display(input_check)

# Allow only the intended queue fields.
allowed_columns = {
    "rank", "client_hash_id", "content_hash_id",
    "observed_days", "impressions", "clicks", "ctr",
    "sessions", "engagement_rate", "avg_position",
    "position_bucket", "volume_bucket", "peer_count",
    "peer_median_ctr", "ctr_gap", "score",
    "reason_code", "action", "confidence_note"
}
assert set(queue.columns) == allowed_columns

# Confirm the output CSV was written successfully.
saved_queue = pd.read_csv(csv_path)
assert len(saved_queue) == len(queue)
assert saved_queue["rank"].tolist() == queue["rank"].tolist()
assert np.allclose(saved_queue["score"], queue["score"])

receipt = {
    "assignment": "ML-07",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "FlyRank/internship-warehouse",
    "input_window": ["2026-03-01", "2026-03-20"],
    "availability": "GSC and GA4 flags both TRUE",
    "minimum_observed_days": 10,
    "minimum_impressions": 100,
    "minimum_peer_items": 20,
    "position_edges": [0, 3, 10, 20, "infinity"],
    "volume_groups": ["100-499", "500-1999", "2000+"],
    "score_formula": (
        "impressions * max(peer_median_ctr - ctr, 0)"
    ),
    "signal_1_verdict": "CONFIRMED",
    "signal_2_verdict": "OPPOSITE",
    "queue_items": int(len(queue)),
    "positive_score_items": int((queue["score"] > 0).sum()),
    "top_k_reviewed": 10,
    "human_review_precision_at_10": None,
    "human_review_label_base_rate": None,
    "evaluation_status": "Independent review labels unavailable",
    "csv_path": str(csv_path),
    "later_outcomes_used_in_score": False,
    "product_flags_used_in_score": False
}

receipt_path = output_dir / "baseline_run_receipt.json"
receipt_path.write_text(
    json.dumps(receipt, indent=2),
    encoding="utf-8"
)

print("Queue field and CSV checks passed.")
print("Run receipt written:", receipt_path)


,first_date,last_date,available_daily_rows
0,2026-03-01,2026-03-20,208242


Queue field and CSV checks passed.
Run receipt written: work/outputs/baseline_run_receipt.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.